# Text Representation

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [3]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [6]:
import pandas as pd
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [9]:
tfidf_vectorizer = TfidfVectorizer() # Create a CountVectorizer instance
vector_matrix = tfidf_vectorizer.fit_transform(data) # Fit and transform the documents into numerical vectors

In [10]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


In [7]:
import pandas as pd
import nltk
import numpy as np
import gensim

from nltk.tokenize import word_tokenize

In [9]:
from pathlib import Path
import pandas as pd

news_csv_candidates = [Path('True.csv'), Path('dataset') / 'True.csv', Path('NLP_Lab5') / 'dataset' / 'True.csv']
simpsons_csv_candidates = [
    Path('dataset') / 'simpsons_script_lines.csv',
    Path('NLP_Lab5') / 'dataset' / 'simpsons_script_lines.csv',
]
csv_path = next((path for path in news_csv_candidates if path.is_file()), None)
if csv_path is None:
    csv_path = next((path for path in simpsons_csv_candidates if path.is_file()), None)
    if csv_path is None:
        raise FileNotFoundError(
            'Place True.csv or dataset/simpsons_script_lines.csv beside the notebook '
            'or in its dataset folder.'
        )
    print('True.csv not found; using the local Simpsons script dataset for Word2Vec.')

df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')
if 'text' not in df.columns and 'spoken_words' in df.columns:
    df = df.rename(columns={'spoken_words': 'text'})
df = df.dropna(subset=['text'])
df.head()

True.csv not found; using the local Simpsons script dataset for Word2Vec.


,raw_character_text,text
0,Miss Hoover,"No, actually, it was a little of both. Sometim..."
1,Lisa Simpson,Where's Mr. Bergstrom?
2,Miss Hoover,I don't know. Although I'd sure like to talk t...
3,Lisa Simpson,That life is worth living.
4,Edna Krabappel-Flanders,The polls will be open from now until the end ...


In [10]:
tokens = []

for i in df['text']:
    token = word_tokenize(str(i).lower())
    tokens.append(token)

In [11]:
w2v = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [12]:
print("Cosine similarity between 'school' and 'program' - Skip Gram : ",w2v.wv.similarity('school', 'program'))

Cosine similarity between 'school' and 'program' - Skip Gram :  0.6907654


In [13]:
print("words that similar to 'program' - Skip Gram : ",w2v.wv.most_similar('program'))

words that similar to 'program' - Skip Gram :  [('presentation', 0.8905940055847168), ('official', 0.8899048566818237), ('physics', 0.8775283098220825), ('bulletin', 0.8767876029014587), ('weekly', 0.8744000792503357), ('foreign', 0.8725759387016296), ('social', 0.8712686896324158), ('assembly', 0.8712176084518433), ('candidate', 0.8704466819763184), ('sacred', 0.8698301911354065)]


# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sentences = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?'
]

sentence_vectors = TfidfVectorizer().fit_transform(sentences)
similarity_matrix = cosine_similarity(sentence_vectors)
pd.DataFrame(similarity_matrix, index=sentences, columns=sentences).round(3)

### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

sentences = [
    'data science is one of the most important fields of science',
    'this is one of the best data science courses',
    'data scientists analyze data'
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(sentences)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f'Sentence {i}' for i in range(1, len(sentences) + 1)]
)
display(tfidf_df.round(3))

for sentence_name, scores in tfidf_df.iterrows():
    important_words = scores[scores > 0].sort_values(ascending=False)
    print(f'{sentence_name} important words: {important_words.index.tolist()}')

## Word2vec

### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)

In [ ]:
from pathlib import Path
import re
import pandas as pd
from gensim.models import Word2Vec

dataset_candidates = [
    Path('dataset') / 'simpsons_script_lines.csv',
    Path('NLP_Lab5') / 'dataset' / 'simpsons_script_lines.csv',
]
dataset_path = next((path for path in dataset_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError(
        'Could not find dataset/simpsons_script_lines.csv. Place it in the project dataset folder.'
    )

df = pd.read_csv(dataset_path, usecols=['spoken_words']).dropna(subset=['spoken_words'])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[0-9]', '', text)
    text = re.sub(r'[)(,.”“.’$-]', '', text)
    return text

tokens = [clean_text(text).split() for text in df['spoken_words']]
tokens = [sentence for sentence in tokens if sentence]

Skip_gram_model = Word2Vec(
    sentences=tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1,
    workers=1,
    seed=42
)
print(f'Trained Skip-gram model on {len(tokens)} Simpsons lines.')

### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



In [ ]:
for character in ['homer', 'marge', 'bart']:
    print(f'Words similar to {character}:')
    print(Skip_gram_model.wv.most_similar(character, topn=10))

### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

In [ ]:
word_groups = [
    ['jimbo', 'milhouse', 'kearney'],
    ['nelson', 'bart', 'milhouse'],
    ['homer', 'patty', 'selma']
]

for words in word_groups:
    print(f'Odd one out in {words}: {Skip_gram_model.wv.doesnt_match(words)}')